# dialog

> Messages, dialogs, and attachments: the data model for LLM conversations stored as notebooks

In [ ]:
#| default_exp dialog

A dialog is a conversation with an LLM kept in a Jupyter notebook: markdown notes, runnable code cells with their outputs, and prompt/response pairs, all in one document that can be viewed, edited, diffed, and version-controlled like any other notebook. The format comes from [Solveit](https://solve.it.com), where people have been editing dialogs as their daily working medium for a couple of years; this library extracts the format and its data model so that other tools can read, build, and surgically edit dialogs too, and convert them to and from LLM chat history.

This module is the in-memory model: `Message`, `Dialog`, and `Attachment`. Reading and writing the `.ipynb` form, and converting dialogs to chat history, live in later modules.

In [ ]:
#| export
import base64, copy, random, weakref
from json import loads, dumps
from fastcore.utils import *

In [ ]:
from fastcore.test import *

## Message types

Every message is one of four types. A `note` is markdown, a `code` message is executable and carries Jupyter outputs, a `prompt` holds a user request together with the AI response it produced, and `raw` is untyped text. These map onto notebook cell types when a dialog is written to disk.

In [ ]:
#| export
smsg_types = scode,snote,sprompt,sraw = "code","note","prompt","raw"

## Identity

Messages, dialogs, and attachments all carry a string id, and it's convenient for code to treat an object and its id interchangeably: `msg in dlg.messages`, `atts.remove('img1.png')`, or comparing a message against a bare id string. `add_id_hash` adds `__eq__` and `__hash__` to a class based on one identifier attribute, so equality and set/dict membership follow the id.

In [ ]:
#| export
def add_id_hash(cls, attr):
    'Adds `__eq__` and `__hash__` to a cls based on a str or int `attr` id'
    @patch
    def __eq__(self:cls, a):
        if a is None: return False
        if isinstance(a, cls): return getattr(self, attr)==getattr(a, attr)
        if isinstance(a, (str, int)): return getattr(self, attr)==a
        return False
    @patch
    def __hash__(self:cls): return hash(getattr(self, attr))

A tiny class shows the behavior. Before patching, neither `in` nor `==` sees two instances with the same id as equal:

In [ ]:
class Type:
    def __init__(self, id): store_attr()

atts = [Type(str(i)) for i in range(5)]
test_eq(Type('0') in atts, False)
test_eq(Type('3') == atts[3], False)

After patching, instances compare by id, ids themselves compare against instances, and hashing follows suit so sets and dict keys behave:

In [ ]:
add_id_hash(Type, 'id')
test_eq(Type('0') in atts, True)
test_eq(Type(10) in atts, False)
test_eq(Type('3') == atts[3], True)
test_eq('1' in atts, True)
test_eq(Type('4') == '4', True)

In [ ]:
atts.remove('0')
test_eq(len(atts), 4)
atts.remove(Type('3'))
test_eq(len(atts), 3)
test_eq([o.id for o in atts], ['1','2','4'])
test_eq(len({Type(1), Type(2), Type(1)}), 2)
test_eq({Type(1):"data"}[Type(1)], "data")

## Dialog

## Msgs

Lists of messages display as one bounded preview line per message (the repr caps at 20 rows; `show` gives explicit control). Rows are the live `Message` objects, so results of searches can be edited directly.

In [ ]:
#| export
class Msgs(L):
    "A list of `Message`s with a preview-per-line repr"
    def show(self,
        maxlen:int=120, # Maximum characters per preview line
        rows:int=None, # Max messages to show (None: all)
    ):
        "One `preview` line per message"
        xs = self if rows is None else self[:rows]
        res = '\n'.join(m.preview(maxlen) for m in xs)
        if rows is not None and len(self)>rows: res += f'\n…({len(self)-rows} more)'
        return PrettyString(res)

    def __repr__(self): return str(self.show(rows=20))

A `Dialog` consists of a name, an ordered list of messages, and `meta` (the notebook-level metadata dict).

In [ ]:
#| export
class Dialog(BasicRepr):
    "A named, ordered list of messages"
    def __init__(self,
        name, # Dialog name, usually the file stem
        messages=None, # Initial `Message`s
        meta=None, # Notebook-level metadata dict, carried verbatim through save/load
    ): self.name,self.messages,self.meta = str(name),Msgs(listify(messages)),dict(meta or {})

    def _repr_markdown_(self):
        msgs = ''
        if self.messages:
            msgr = '\n'.join(f'- {o.summ}' for o in self.messages)
            msgs = f'''
<details markdown='1'>

{msgr}

</details>'''
        return f'''
**{self.name}**

{msgs}'''

add_id_hash(Dialog, 'name')

In [ ]:
dlg = Dialog('test')
dlg

<div class="prose" markdown="1">


**test**



</div>

## Message

Prompt and code outputs both use the same ipynb-compatible structure: a list of output dicts with `output_type`, `data`, and `metadata` fields. One output format means one rendering and serialization path for both.

For prompts, the AI response text is stored in `data['text/markdown']` with `metadata['is_ai_res']=True` to distinguish it from other display outputs. The `ai_res` property on `Message` extracts this text.

In [ ]:
#| export
def mk_output(typ, d, meta=None, **kw): return dict(output_type=typ, metadata=meta or {}, data=d, **kw)
def mk_displayobj(d, meta=None): return mk_output('display_data', d, meta)
def displayobj(data='', subtype='plain', mimetype='text', meta=None): return mk_displayobj({f'{mimetype}/{subtype}': data}, meta)
def mk_code_output(d): return [mk_output('execute_result', d, execution_count=1)]
def code_output(result='', subtype='plain', mimetype='text'): return mk_code_output({f'{mimetype}/{subtype}': result})

def prompt_output(txt=''):
    "Create prompt output list from AI response text"
    return [displayobj(txt, 'markdown', meta={'is_ai_res': True})] if txt else []

A `Message` carries exactly what the ipynb spec provides for a cell: `content` (source), `output`, `msg_type` (cell type, plus the prompt convention), `id`, `attachments`, and `meta` — the cell's metadata dict, carried verbatim through save/load, so annotations this library knows nothing about survive a round trip untouched. Hosts declare the metadata they want promoted to real attributes in `meta_attrs` (attribute name → metadata key); anything undeclared simply rides along in `meta`. Ids are `_` plus four hex bytes so they're valid DOM ids. `content`, `output`, and `msg_type` are `DepProp` descriptors that invalidate cached renders on assignment, through the `clear_out_cache`/`clear_inp_cache` hooks: core's only cache is `_ai_rend` (the for-AI rendering), and a subclass that memoizes more overrides the hooks to clear its own slots at the same moments. The `dlg` backlink is a weakref to avoid a circular reference.

In [ ]:
#| export
class Message:
    meta_attrs = {} # Attribute name -> cell metadata key: fields a subclass promotes from cell metadata and serializes back

    def __init__(self,
        content='', # The message text: markdown, code, or a prompt's request
        dlg=None, # The `Dialog` this message belongs to
        output='', # Jupyter-style output list (code and prompt messages), or ''
        id=None, # Message id; `_` plus 4 random hex bytes if None
        msg_type='code', # One of `smsg_types`: code, note, prompt, or raw
        attachments=None, # `Attachment`s carried by the message
        meta=None, # Remaining cell metadata, carried verbatim through save/load
        **xtras, # Extra attributes to store as-is
    ):
        "An ipynb cell, or Solveit message (which adds the `prompt` type)"
        # Prepend `_` to ensure it is a valid DOM id too
        if id is None: id = '_'+rtoken_hex(4)
        attachments = listify(attachments)
        meta = dict(meta or {})
        if content  is UNSET: content = ''
        if msg_type is UNSET: msg_type = snote
        if output   is UNSET: output=[] if msg_type in (scode, sprompt) else ''
        self.msg_type = msg_type # Set type first, since it clears output
        self._dlg = weakref.ref(dlg) if dlg else None # weakref to avoid circular ref
        store_attr(but=['dlg', 'msg_type', 'xtras'])
        for k,v in xtras.items(): setattr(self, k, v)

    # Invalidation hooks, called when output/content/type change; subclasses clear their extra caches here
    def clear_out_cache(self): self._ai_rend = None
    def clear_inp_cache(self): pass

    @DepProp
    def output(self): self.clear_out_cache()

    @output.norm
    def output(self, v):
        if self.msg_type==sprompt and isinstance(v, str): return prompt_output(v)
        return [] if v is None and self.msg_type in (scode, sprompt) else v

    @DepProp
    def content(self): self.clear_inp_cache()

    @DepProp
    def msg_type(self):
        self.clear_out_cache()
        self.clear_inp_cache()
        self.output = [] if self.msg_type in (scode, sprompt) else ''

    @property
    def ai_res(self):
        "Extract AI response text from prompt output list"
        if self.msg_type != sprompt: return ''
        if isinstance(self.output, str): return self.output  # Handle legacy/direct string
        if res := first(L(self.output).filter(lambda x: nested_idx(x, 'metadata', 'is_ai_res'))):
            return nested_idx(res, 'data', 'text/markdown') or ''
        return ''

    def __deepcopy__(self, memo):
        cls = self.__class__
        result = cls.__new__(cls)
        memo[id(self)] = result
        for k, v in self.__dict__.items():
            if k.startswith(('_msgidle','_dlg')): setattr(result, k, v)
            else: setattr(result, k, copy.deepcopy(v, memo))
        return result

    @property
    def dlg(self): return self._dlg() if self._dlg else None
    @dlg.setter
    def dlg(self, v): self._dlg = weakref.ref(v)
    @property
    def __flds__(self): return vars_pub(self) + ['content', 'output', 'msg_type']
    @property
    def flds(self): return [o for o in self.__flds__ if o not in ('content', 'output', 'attachments')]
    @property
    def summ(self): return truncstr(self.content, 96) + (f" ⇒ {truncstr(str(self.output), 60)}" if self.output else '')
    __repr__ = basic_repr('id,content,output,msg_type')

In [ ]:
#| export
add_id_hash(Message, 'id')
Dialog.msg_cls = Message

`Dialog.msg_cls` is assigned here so that every factory below constructs messages through it: a subclassed `Dialog` sets `msg_cls` to its own `Message` subclass and gets the right instances from all inherited code, and `read_ipynb` takes a `cls` so whole files load as a host's types. 

The repr and markdown display show the content, the output when there is one, and the flags:

In [ ]:
#| export
@patch
def _repr_markdown_(self:Message):
    detls = '\n'.join(f'- {k}: {getattr(self, k)}' for k in self.flds)
    return f"""{self.summ}

<details>

{detls}

</details>"""

In [ ]:
m = Message('in', output='out', dlg=dlg, msg_type=snote)
m

<div class="prose" markdown="1">

in ⇒ out

<details>

- id: _ef8642b4
- meta: {}
- msg_type: note

</details>

</div>

Prompt outputs normalize on the way in: a string assigned to a prompt message's `output`, at construction or any time later, wraps through `prompt_output`, so `ai_res` and history rendering always see the one list format.

In [ ]:
pm = Message('q', msg_type=sprompt, output='A reply')
test_eq(pm.ai_res, 'A reply')
pm.output = 'Edited reply'
test_eq(pm.output, prompt_output('Edited reply'))
test_eq(pm.ai_res, 'Edited reply')

In [ ]:
#| export
@patch
def preview(self:Message,
    maxlen:int=120, # Maximum characters in the line
):
    "Escaped one-line summary of this message, the row format for `Msgs` displays"
    s = f"{self.id}:{self.msg_type}: {self.content}"
    if self.msg_type==sprompt and self.ai_res: s += f" ⇒ reply({len(self.ai_res)})"
    elif self.output: s += f" ⇒ out({len(str(self.output))})"
    return truncstr(s.replace('\n', '\\n'), maxlen)

`preview` is what `Msgs` renders per row: id, type, escaped content start, and a size hint for any reply or output. `summ` (used by the markdown reprs) is bounded the same way, so a message with a huge output can never flood a display:

In [ ]:
pv = pm.preview()
assert pv.startswith(f'{pm.id}:prompt: q') and 'reply(12)' in pv
big = Message('x=1', msg_type=scode, output=code_output('y'*500))
assert 'out(' in big.preview() and len(big.summ) < 200
test_eq(len(Message('z'*300, msg_type=snote).preview(80)), 80)
dd = Dialog('t', [pm, big])
assert isinstance(dd.messages, Msgs)
test_eq(str(dd.messages.show(rows=1)).count('\n'), 1)  # one row plus the "more" marker

`update` assigns several attributes at once, skipping any left as `UNSET`, which is what route handlers and edit operations pass for fields the user didn't touch:

In [ ]:
#| export
@patch
def update(self:Message, **kwargs):
    "Update message attributes with provided keyword arguments"
    for k, v in kwargs.items():
        if v is not UNSET: setattr(self, k, v)
    return self

## Building dialogs

`mk_message` creates a message through `msg_cls` and inserts it at the start, end, or relative to another message (which may be given as a `Message` or an id):

In [ ]:
#| export
@patch
def mk_message(self:Dialog,
    content:str, # Message text
    idx=-1, # Insert position; -1 appends
    after=None, # Insert after this `Message` or id
    before=None, # Insert before this `Message` or id
    msg_type=scode, # One of `smsg_types`
    output='', # Output list; code messages also accept a JSON string
    **kwargs, # Passed to `msg_cls`
):
    "Make new message and insert it into notebook before/after specified cell, or start of list (idx=0) by default"
    if msg_type==scode and isinstance(output,str): output = loads(output or '[]')
    msg = self.msg_cls(content, self, msg_type=msg_type, output=output, **kwargs)
    if isinstance(after,  Message): after =after.id
    if isinstance(before, Message): before=before.id
    if after is not None: idx = next((i for i,c in enumerate(self.messages) if c.id==after ), -1)+1
    elif before is not None: idx = next((i for i,c in enumerate(self.messages) if c.id==before), -1)
    if idx==-1: idx=len(self.messages)
    self.messages.insert(idx, msg)
    return msg

In [ ]:
dlg = Dialog('test')
msg = dlg.mk_message(content='1')
dlg.mk_message(content='0', before=msg) # 0 before 1
dlg.mk_message(content='2', after=msg) # 2 after 1
assert L(dlg.messages).attrgot('content') == ['0','1','2']

Outputs attach at creation time; `code_output` builds a plausible one by hand:

In [ ]:
out = code_output('64')
test_msg = dlg.mk_message('8*8', output=out)
test_eq(test_msg.output, out)
test_msg

<div class="prose" markdown="1">

8*8 ⇒ [{'output_type': 'execute_result', 'metadata': {}, 'data': {'text/plain': '64'}, 'execution_count': 1}]

<details>

- id: _126a40a4
- meta: {}
- msg_type: code

</details>

</div>

When pasting messages from a clipboard, we need to copy each message and place the copies one after the other at a specific point. `mk_messages` makes that convenient:

In [ ]:
#| export
@patch
def mk_messages(self:Dialog, msgs, after=None, before=None):
    "Make new messages and insert them sequentially into the notebook after/before a specific message."
    def _kws(m): return {o: getattr(m,o) for o in m.flds if o!='id'}
    return [after:=self.mk_message(m.content,output=m.output,attachments=m.attachments,after=after, before=before, **_kws(m)) for m in msgs]

Mimicking copy/paste: copy the first two messages, then paste them after the second one.

In [ ]:
dlg = Dialog('copy_paste')
for i in '123': dlg.mk_message(i, msg_type=snote)
def _contents(dlg): return [m.content for m in dlg.messages]
clipboard = copy.copy(dlg.messages[:2])
dlg.mk_messages(clipboard, after=dlg.messages[1])
test_eq(_contents(dlg), ['1', '2', '1', '2', '3'])

In [ ]:
#| export
@patch
def insert_after(self:Message, msgs):
    curr = self.dlg.messages
    idx = curr.index(self)
    curr[idx+1:idx+1] = msgs
    for m in msgs: m.dlg = self.dlg

In [ ]:
dlg = Dialog('test')
msg1,msg2,msg3 = [dlg.mk_message(o) for o in ('first','second','third')]
msg1.insert_after([Message('inserted1'), Message('inserted2')])
test_eq([m.content for m in dlg.messages], ['first', 'inserted1', 'inserted2', 'second', 'third'])

## Navigation

`next`, `previous`, and `rel_msg` find the adjacent message in either direction.

In [ ]:
#| export
@patch
def _neighbor(self:Message, fwd=True, pred=None):
    "Find next (fwd=True) or previous (fwd=False) message, optionally filtered by pred."
    msgs = self.dlg.messages
    idx = msgs.index(self)
    slc = msgs[idx+1:] if fwd else reversed(msgs[:idx])
    return first(m for m in slc if not pred or pred(m))

@patch
def next(self:Message): return self._neighbor()
@patch
def previous(self:Message): return self._neighbor(fwd=False)
@patch
def rel_msg(self:Message, is_up:bool): return self._neighbor(fwd=not is_up)

In [ ]:
test_eq(msg1.next().content, 'inserted1')
test_eq(msg2.previous().content, 'inserted2')
test_eq(msg3.next(), None)
test_eq(msg1.previous(), None)

`find_msg` and `get_msg` look messages up by content or id:

In [ ]:
#| export
@patch
def find_msg(self:Dialog, content): return first(m for m in self.messages if m.content==content)

def get_msg(id_, dlg):
    if not dlg or not id_: return
    return first(o for o in dlg.messages if o.id==id_)

In [ ]:
test_eq(dlg.find_msg('second'), msg2)
test_eq(get_msg(msg2.id, dlg), msg2)
test_eq(get_msg('nope', dlg), None)

Removing takes messages out of the list and hands them back, which is what cut and delete both need:

In [ ]:
#| export
@patch
def remove_msgs(self:Dialog, msgs):
    "Remove messages from dialog, return removed list"
    s = set(msgs)
    self.messages[:] = [m for m in self.messages if m not in s]
    return msgs

@patch
def remove_from_dlg(self:Message):
    "Remove this message from its dialog"
    return self.dlg.remove_msgs([self])

In [ ]:
dlg = Dialog('test_remove')
m1, m2, m3 = [dlg.mk_message(f'msg{i}') for i in range(3)]
removed = dlg.remove_msgs([m1, m3])
test_eq(dlg.messages, [m2])
test_eq(removed, [m1, m3])
m2.remove_from_dlg()
test_eq(dlg.messages, [])

## Inspecting outputs

`has_error` reports whether a code message's outputs include an error, and `get_output_mds` pulls out any markdown an output displayed.

In [ ]:
#| export
@patch(as_prop=True)
def has_error(self:Message):
    if self.msg_type != scode: return False
    return any(o.get('output_type') == 'error' for o in (self.output or []))

def get_output_mds(output):
    "Extract text/markdown content from code outputs (display_data and execute_result)"
    if not output: return []
    return [o['data']['text/markdown'] for o in output
        if o.get('output_type') in ('display_data', 'execute_result') and 'text/markdown' in o.get('data', {})]

In [ ]:
err_out = [dict(output_type='error', ename='ZeroDivisionError', evalue='division by zero', traceback=[])]
test_eq(Message('1/0', msg_type=scode, output=err_out).has_error, True)
test_eq(Message('1+1', msg_type=scode, output=code_output('2')).has_error, False)
test_eq(Message('note', msg_type=snote).has_error, False)

Only display and result outputs carry markdown; streams (prints) do not:

In [ ]:
out_stream = [dict(output_type='stream', name='stdout', text='plain\n')]
out_display = [dict(output_type='display_data', metadata={}, data={'text/markdown': '*hi*'})]
out_exec = [dict(output_type='execute_result', metadata={}, data={'text/markdown': '**lo**'}, execution_count=1)]
test_eq(get_output_mds(out_stream), [])
test_eq(get_output_mds(out_display + out_exec), ['*hi*', '**lo**'])
test_eq(get_output_mds(None), [])

Merging combines several messages into one; `merge_content` is each message's contribution, in terms the *target* type understands: merging into code keeps sources raw, while merging into a note renders a prompt as its request/response sections and fences code. Hosts override it for their own conventions (llmsurgery's `merge_msgs` drives it):

In [ ]:
#| export
@patch
def merge_content(self:Message,
    mtype:str, # The merge target's msg_type
):
    "This message's content contribution when merged into a `mtype` message; hosts override for their own conventions"
    if mtype==scode: return self.content
    if self.msg_type==sprompt: return f'## AI Prompt\n{self.content}\n## AI Response\n{self.ai_res}'
    if self.msg_type==scode: return f'```python\n{self.content}\n```'
    if self.msg_type==sraw: return f'```plaintext\n{self.content}\n```'
    return self.content

In [ ]:
pm2 = Message('Sum?', msg_type=sprompt, output='It is 3.')
test_eq(pm2.merge_content(snote), '## AI Prompt\nSum?\n## AI Response\nIt is 3.')
test_eq(pm2.merge_content(scode), 'Sum?')
test_eq(Message('x=1', msg_type=scode).merge_content(snote), '```python\nx=1\n```')

## Validation

Serialization builds valid cells by construction, but nothing stops code from assigning a malformed output list, an unserializable `meta` value, or a duplicate id directly; such damage otherwise only surfaces as a confusing error at save time. `validate` raises a `ValueError` naming the message as soon as it's called, and never repairs. These are the model-level rules; file-level structure is `fastcore.nbio`'s `validate_nb`'s business.

In [ ]:
#| export
_out_types = {'stream','display_data','execute_result','error'}

@patch
def validate(self:Message):
    "Raise `ValueError` for problems that would break saving or rendering; returns self if fine"
    if self.msg_type not in smsg_types: raise ValueError(f"{self.id}: unknown msg_type {self.msg_type!r}")
    if not (self.id and isinstance(self.id, str)): raise ValueError(f"message missing id: {truncstr(self.content, 30)!r}")
    if not isinstance(self.content, str): raise ValueError(f"{self.id}: content must be str")
    o = self.output
    if self.msg_type in (scode, sprompt):
        if not isinstance(o, list): raise ValueError(f"{self.id}: {self.msg_type} output must be a list")
        if bad := first(x for x in o if not (isinstance(x, dict) and x.get('output_type') in _out_types)):
            raise ValueError(f"{self.id}: bad output entry {truncstr(str(bad), 50)!r}")
    elif o: raise ValueError(f"{self.id}: {self.msg_type} message must not carry output")
    try: dumps(self.meta)
    except TypeError as e: raise ValueError(f"{self.id}: meta not JSON-serializable: {e}") from None
    return self

In [ ]:
m = Message('1+1', msg_type=scode, output=code_output('2'))
test_eq(m.validate(), m)
m.output = [dict(no_type=1)]
with expect_fail(ValueError, 'bad output entry'): m.validate()
with expect_fail(ValueError, 'must not carry output'): Message('n', msg_type=snote, output='x').validate()
with expect_fail(ValueError, 'not JSON-serializable'): Message('m', msg_type=snote, meta=dict(x={1,2})).validate()

In [ ]:
#| export
@patch
def validate(self:Dialog):
    "Validate every message plus dialog-scope rules (unique ids, serializable `meta`); returns self if fine"
    ids = [m.id for m in self.messages]
    if dups := {i for i in ids if ids.count(i)>1}: raise ValueError(f"duplicate message id(s): {', '.join(sorted(dups))}")
    for m in self.messages: m.validate()
    try: dumps(self.meta)
    except TypeError as e: raise ValueError(f"dialog meta not JSON-serializable: {e}") from None
    return self

In [ ]:
vd = Dialog('v', [Message('a', msg_type=snote), Message('b', msg_type=snote)])
test_eq(vd.validate(), vd)
vd.messages.append(Message('c', msg_type=snote, id=vd.messages[0].id))
with expect_fail(ValueError, 'duplicate message id'): vd.validate()

## Attachments

Attachments are named binary blobs (usually images) that ride along with a message and serialize into the standard Jupyter attachments dict. Ids follow the uuid4 shape; `ruuid4` builds them from `rtoken_hex` so tests that seed `random` get deterministic ids.

In [ ]:
#| export
def ruuid4():
    "Generate a deterministic UUID4-like string using rtoken_hex"
    parts = [rtoken_hex(4), rtoken_hex(2), rtoken_hex(2), rtoken_hex(2), rtoken_hex(6)]
    parts[2], parts[3] = '4' + parts[2][1:], format(random.randint(8, 11), 'x') + parts[3][1:]
    return f"{parts[0]}-{parts[1]}-{parts[2]}-{parts[3]}-{parts[4]}"

class Attachment(BasicRepr):
    def __init__(self, data:bytes, content_type:str, id:str=''):
        if not id: id=ruuid4()
        store_attr()

add_id_hash(Attachment, 'id')

@patch
def mk_attachment(self:Message, data:bytes, content_type:str, id:str=''):
    att = Attachment(data=data, content_type=content_type, id=id)
    return self.attachments.append(att)

`tiny_png` is a minimal valid one-pixel PNG, exported for examples and tests that need real image bytes.

In [ ]:
#| export
tiny_png = base64.b64decode('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNkYPhfDwAChwGA60e6kgAAAABJRU5ErkJggg==')

Copy/paste keeps attachments with their messages:

In [ ]:
dlg_attach = Dialog('attach_test')
att1 = Attachment(data=b'screenshot1', content_type='image/png', id='img1.png')
m1 = dlg_attach.mk_message('1. Message with screenshot', msg_type=snote, attachments=[att1])
m2 = dlg_attach.mk_message('2. No attachment', msg_type=snote)
pasted = dlg_attach.mk_messages(copy.copy(dlg_attach.messages[:1]), after=m2)
test_eq(len(dlg_attach.messages), 3)
test_eq(pasted[0].attachments[0].id, 'img1.png')
test_eq(pasted[0].attachments[0].data, b'screenshot1')

## Serializing

In [ ]:
#| export
@patch
def todict(self:Dialog): return {k:v for k,v in asdict(self).items() if k[0]!='_'}

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()